# BIOMASS VV — Corrected SLC stack

Load all seven full-resolution BIOMASS STA acquisitions, reconstruct the **VV** complex SLC (GDAL band 4), apply the precomputed flattening and SKP phase screens, and save both `data/processed/vv_stack.npy` and `data/processed/vv_stack_metadata.json`.

The phase correction uses $S_{\mathrm{corr}} = S\, \exp\big(j(\phi_{\mathrm{flat}}+\phi_{\mathrm{SKP}})\big)$. Run this notebook in the project\'s `biomass-tomosar` environment.

In [ ]:
from pathlib import Path
import numpy as np
from osgeo import gdal
from tqdm.auto import tqdm

from biomass_tomosar.bm_data_loader import bmStack
from biomass_tomosar.paths import biomass_stack_dir, processed_data_dir

POLARIZATION = "VV"
BAND_INDEX = 4  # TIFF bands: HH=1, HV=2, VH=3, VV=4

products = sorted(
    p for p in biomass_stack_dir().iterdir()
    if p.is_dir() and p.name.startswith("BIO_S2_STA__1S_")
)
if len(products) != 7:
    raise ValueError(f"Expected 7 STA acquisitions, found {len(products)} in {biomass_stack_dir()}")

stack = bmStack([str(p) for p in products])
output_dir = processed_data_dir()
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Acquisitions: {len(products)}")
print(f"Image shape: ({stack.nli}, {stack.nsa})")
print(f"Output directory: {output_dir}")

In [ ]:
def read_slc(amp_ds, phase_ds, band_index):
    amp_band = amp_ds.GetRasterBand(band_index)
    phase_band = phase_ds.GetRasterBand(band_index)
    amplitude = amp_band.ReadAsArray()
    phase = phase_band.ReadAsArray()

    invalid = ~np.isfinite(amplitude) | ~np.isfinite(phase)
    for array, raster_band in ((amplitude, amp_band), (phase, phase_band)):
        nodata = raster_band.GetNoDataValue()
        if nodata is not None:
            invalid |= array == nodata

    product_nodata = getattr(stack, "noDataValue", None)
    if product_nodata is not None:
        invalid |= (amplitude == product_nodata) | (phase == product_nodata)

    slc = (amplitude * np.exp(1j * phase)).astype(np.complex64)
    slc[invalid] = 0
    return slc


slc_stack = np.empty((stack.Ntracks, stack.nli, stack.nsa), dtype=np.complex64)

for i, (product, lut) in enumerate(tqdm(zip(products, stack.luts), total=stack.Ntracks)):
    stem = product.name[:70].lower()
    amp_path = product / "measurement" / f"{stem}_i_abs.tiff"
    phase_path = product / "measurement" / f"{stem}_i_phase.tiff"

    amp_ds = gdal.Open(str(amp_path), gdal.GA_ReadOnly)
    phase_ds = gdal.Open(str(phase_path), gdal.GA_ReadOnly)
    if amp_ds is None or phase_ds is None:
        raise FileNotFoundError(f"Could not open {amp_path} or {phase_path}")

    # Reconstruct the complete complex image from amplitude and phase.
    slc = read_slc(amp_ds, phase_ds, BAND_INDEX)
    amp_ds = phase_ds = None

    # Interpolate the precomputed phase-screen LUTs to the SLC grid.
    flat = np.ma.filled(
        lut.get("coregistration", "flatteningPhaseScreen").upsampled, np.nan
    )
    skp = np.ma.filled(
        lut.get("skpPhaseCalibration", "skpCalibrationPhaseScreen").upsampled, np.nan
    )
    if flat.shape != slc.shape or skp.shape != slc.shape:
        raise ValueError(f"Phase screen and SLC shapes differ for acquisition {i + 1}")

    total_phase = flat + skp
    invalid_phase = (
        ~np.isfinite(total_phase) | (flat == -9999) | (skp == -9999)
    )
    total_phase[invalid_phase] = 0

    # BIOMASS correction convention: exp(+j * (flattening + SKP)).
    slc *= np.exp(1j * total_phase).astype(np.complex64)
    slc[invalid_phase] = 0
    slc_stack[i] = slc

    print(f"{POLARIZATION}: acquisition {i + 1}/7 complete")

# Persist the corrected stack in the TOML-configured processed-data folder.
output_file = output_dir / f"{POLARIZATION.lower()}_stack.npy"
np.save(output_file, slc_stack, allow_pickle=False)

print(f"\nSaved: {output_file}")
print(f"Shape: {slc_stack.shape}, dtype: {slc_stack.dtype}")
print(f"File size: {output_file.stat().st_size / 2**30:.2f} GiB")

In [ ]:
# Save metadata alongside the corrected SLCs, for later processing and alignment.
import json

MASTER_IDX = 3  # Fourth acquisition is the STA reference
annot = stack.annot_coreg[MASTER_IDX]


def xml_value(name):
    nodes = annot.xpath(
        f"//*[local-name()='sarImage']/*[local-name()='{name}']"
    )
    return nodes[0].text.strip() if nodes and nodes[0].text else None


metadata = {
    "shape": list(slc_stack.shape),
    "dtype": str(slc_stack.dtype),
    "axis_order": ["acquisition", "azimuth", "range"],
    "polarization": POLARIZATION,
    "phase_corrections": ["flattening", "SKP"],
    "phase_correction_exponent_sign": "+",
    "reference_acquisition_index": MASTER_IDX,
    "acquisitions": [p.name for p in products],
    "wavelength_m": float(stack.lamb),
    "grid": {
        "number_of_azimuth_lines": int(stack.nli),
        "number_of_range_samples": int(stack.nsa),
        "azimuth_pixel_spacing_m": xml_value("azimuthPixelSpacing"),
        "range_pixel_spacing_m": xml_value("rangePixelSpacing"),
        "azimuth_time_interval_s": xml_value("azimuthTimeInterval"),
        "range_time_interval_s": xml_value("rangeTimeInterval"),
        "projection": xml_value("projection"),
    },
}

metadata_file = output_dir / f"{POLARIZATION.lower()}_stack_metadata.json"
with metadata_file.open("w") as f:
    json.dump(metadata, f, indent=4)

print(f"Metadata saved: {metadata_file}")
